# 🎓 University Final Project: 2D Image to 3D Model AI Backend
### Fast 360° 3D Mesh Reconstruction Server powered by Google Colab GPU (T4)

---
### 📌 Instructions:
1. Ensure you are connected to a **GPU Runtime**: 
   - Click **Runtime** -> **Change runtime type** -> select **T4 GPU** -> **Save**.
2. Click **Runtime** -> **Run all** (or run each cell sequentially).
3. Cell 4 will output a public HTTPS URL (e.g. `https://xxxx.trycloudflare.com`).
4. Copy and paste that URL into your **3D Vision Studio** Web App settings!

In [ ]:
# Step 1: Verify NVIDIA GPU & Install Dependencies
!nvidia-smi

print("[*] Installing dependencies (PyTorch, TripoSR, rembg, FastAPI, Cloudflared)... please wait ~1-2 minutes...")
!pip install -q fastapi uvicorn python-multipart rembg trimesh einops omegaconf transformers
!pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git

# Download cloudflared tunnel binary for instant public HTTPS endpoint
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("[+] All packages and tools installed successfully!")

In [ ]:
# Step 2: Load TripoSR Model and rembg Background Removal Session
import torch
import rembg
from tsr.system import TSR

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"[*] Initializing AI Models on: {device}...")

rembg_session = rembg.new_session("u2net")

tsr_model = TSR.from_pretrained(
    "stabilityai/TripoSR",
    config_name="config.yaml",
    weight_name="model.ckpt",
)
tsr_model.renderer.set_chunk_size(8192)
tsr_model.to(device)

print(f"[+] Model loaded successfully on GPU: {torch.cuda.get_device_name(0)}!")

In [ ]:
# Step 3: Define FastAPI Endpoints
import io
import time
from PIL import Image
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from tsr.utils import resize_foreground

app = FastAPI(title="3D Vision Studio API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_alloc = torch.cuda.memory_allocated(0) / (1024 ** 3) if torch.cuda.is_available() else 0.0
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
    return {
        "status": "ok",
        "gpu_name": gpu_name,
        "vram_allocated_gb": round(vram_alloc, 2),
        "vram_total_gb": round(vram_total, 2),
        "model_ready": True,
        "timestamp": time.time()
    }

@app.post("/api/generate")
async def generate_3d(image: UploadFile = File(...)):
    contents = await image.read()
    pil_img = Image.open(io.BytesIO(contents)).convert("RGB")
    
    # 1. Background removal
    input_rgba = rembg.remove(pil_img, session=rembg_session)
    
    # 2. Resize and center foreground
    foreground = resize_foreground(input_rgba, ratio=0.85)
    
    # 3. 3D Neural reconstruction
    with torch.no_grad():
        scene_codes = tsr_model([foreground], device=device)
        meshes = tsr_model.extract_mesh(scene_codes, resolution=256, has_texture=True)
    
    mesh = meshes[0]
    glb_io = io.BytesIO()
    mesh.export(glb_io, file_type="glb")
    
    return Response(
        content=glb_io.getvalue(),
        media_type="model/gltf-binary",
        headers={"Content-Disposition": 'attachment; filename="model.glb"'}
    )

print("[+] FastAPI Application defined.")

In [ ]:
# Step 4: Launch FastAPI Server & Expose via Cloudflare Tunnel
import subprocess
import threading
import time
import re
import uvicorn

# Start Uvicorn in background thread
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print("[+] Uvicorn server listening on port 8000.")

# Launch Cloudflared tunnel
print("[*] Launching secure Cloudflare public tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*60)
    print("🎉 SUCCESS! YOUR COLAB BACKEND IS ONLINE!")
    print(f"👉 COPY THIS URL INTO YOUR WEB APP:\n{tunnel_url}")
    print("="*60 + "\n")
else:
    print("[!] Cloudflare tunnel did not output URL. You can also use ngrok as fallback.")